In [1]:
import os
import sys
import torch
import random
import traceback
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

# If needed, append your alignment_v2 path:
# sys.path.insert(0, "/path/to/alignment_v2")

from alignment_v2.datasets import get_dataset
from alignment_v2.models.registry import get_model
from alignment_v2 import train
from alignment_v2.train import progressive_dropout
from alignment_v2 import plotting

# ---------------------------------------------------------
# Force 'spawn' start method for safety on HPC or macOS:
# ---------------------------------------------------------
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

# ---------------------------------------------------------
# Define ddp_worker at top-level (no nesting)
# ---------------------------------------------------------
def ddp_worker(rank, world_size, port, epochs, batch_size, lr, dropout_rate):
    """
    Each spawned process runs this function.
    The error "Can't get attribute 'ddp_worker'" means ddp_worker
    wasn't accessible at import-time for the child process.
    """
    try:
        # Force IPv4 on localhost port to avoid address family issues
        os.environ["MASTER_ADDR"] = "127.0.0.1"
        os.environ["MASTER_PORT"] = str(port)

        # optional interface override if needed
        # os.environ["NCCL_SOCKET_IFNAME"] = "lo"
        # os.environ["GLOO_SOCKET_IFNAME"] = "lo"

        dist.init_process_group("nccl", rank=rank, world_size=world_size)
        torch.cuda.set_device(rank)
        device = torch.device(f"cuda:{rank}")

        print(f"\n[DDP_RANK {rank}] Init done. device={device}, MASTER_PORT={port}")

        # 1) Build model
        model_name = "AlexNet"
        dataset_name = "ImageNet"
        net = get_model(
            model_name,
            build=True,
            dataset=dataset_name,
            dropout=dropout_rate,
            ignore_flag=False
        ).to(device)

        ddp_net = DDP(net, device_ids=[rank], output_device=rank)
        print(f"[DDP_RANK {rank}] Model & DDP done.")

        # 2) Build dataset w/ distributed=True
        loader_params = dict(batch_size=batch_size, shuffle=False, num_workers=2)
        dataset = get_dataset(
            dataset_name,
            build=True,
            transform_parameters=ddp_net.module,
            loader_parameters=loader_params,
            device="cpu",
            distributed=True
        )
        print(f"[DDP_RANK {rank}] Dataset loaded OK.")

        # 3) Create optimizer
        optimizer = torch.optim.Adam(ddp_net.parameters(), lr=lr, weight_decay=0)

        # 4) Train
        train_params = dict(
            num_epochs=epochs,
            alignment=True,
            alignment_expansion=False,
            compare_expected=False,
            frequency=1,
            delta_alignment=False,
        )
        nets = [ddp_net]
        optimizers = [optimizer]

        print(f"[DDP_RANK {rank}] Starting training epochs={epochs}...")
        results = train.train(nets, optimizers, dataset, **train_params)
        print(f"[DDP_RANK {rank}] Training done. Keys={list(results.keys())}")

        # 5) Progressive Dropout (only rank 0 for plotting)
        if rank == 0:
            print("[DDP_RANK 0] Doing progressive dropout.")
            if "alignment" not in results:
                print("[DDP_RANK 0] No alignment in results, skipping dropout experiment.")
            else:
                drop_params = {"num_drops": 3, "by_layer": False, "train_set": False}
                drop_res = progressive_dropout(nets, dataset, alignment=results["alignment"], **drop_params)
                print("[DDP_RANK 0] dropout results keys:", list(drop_res.keys()))

                # Optional plotting
                plotting.plot_dropout_results(
                    exp=None,
                    dropout_results=drop_res,
                    dropout_parameters=drop_params,
                    prms={
                        "vals": ["AlexNet"],
                        "name": "ModelType",
                        "dataset": "ImageNet",
                        "dropout": dropout_rate,
                        "lr": lr,
                        "weight_decay": 0
                    },
                    dropout_type="alignment-based",
                )

    except Exception as e:
        print(f"[DDP_RANK {rank}] ERROR => {e}")
        traceback.print_exc()
        raise e
    finally:
        dist.destroy_process_group()
        print(f"[DDP_RANK {rank}] Dist group destroyed. Exiting rank {rank}...")

# ---------------------------------------------------------
# Define run_ddp_experiment at top-level
# ---------------------------------------------------------
def run_ddp_experiment(epochs=2, batch_size=64, lr=1e-3, dropout_rate=0.0, n_gpus=4):
    """
    Spawns n_gpus processes, each calling ddp_worker.
    """
    import random
    port = random.randint(20000, 30000)  # random port to avoid collisions
    print(f"DDP launching with n_gpus={n_gpus}, port={port}, epochs={epochs}, batch_size={batch_size}...")

    mp.spawn(
        ddp_worker,
        nprocs=n_gpus,
        args=(n_gpus, port, epochs, batch_size, lr, dropout_rate),
        join=True
    )
    print("DDP experiment finished successfully.")

# ---------------------------------------------------------
# Actually run it
# ---------------------------------------------------------
run_ddp_experiment(
    epochs=2,
    batch_size=64,
    lr=1e-3,
    dropout_rate=0.0,
    n_gpus=4
)

DDP launching with n_gpus=4, port=22766, epochs=2, batch_size=64...


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'ddp_worker' on <module '__main__' (built-in)>
    exitcode = _main(fd, parent_sentinel)
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'ddp_worker' on <module '__main__' (built-in)>
Tracebac

ProcessExitedException: process 3 terminated with exit code 1

In [2]:
# Cell 1: Imports and Global Config

import os
import sys
import torch
import random
import traceback
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

# Insert alignment_v2 path if needed
# sys.path.insert(0, "/path/to/alignment_v2")

from alignment_v2.datasets import get_dataset
from alignment_v2.models.registry import get_model
from alignment_v2 import train
from alignment_v2.train import progressive_dropout
from alignment_v2 import plotting

# HYPERPARAMS - adjust as needed
NUM_GPUS       = 4
EPOCHS         = 2
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
DROPOUT_RATE   = 0.0
DATASET_NAME   = "ImageNet"   # Must exist at dataset path
MODEL_NAME     = "AlexNet"

In [ ]:
def ddp_worker(rank, world_size, port, epochs, batch_size, lr, dropout_rate):
    """
    Each spawned process runs this function.
    """
    try:
        # Force local IPv4 to avoid "address family not supported" issues
        os.environ["MASTER_ADDR"] = "127.0.0.1"
        os.environ["MASTER_PORT"] = str(port)

        # optional: further enforce NCCL / GLOO to pick correct interfaces
        # os.environ["NCCL_SOCKET_IFNAME"] = "lo"
        # os.environ["GLOO_SOCKET_IFNAME"] = "lo"

        dist.init_process_group("nccl", rank=rank, world_size=world_size)
        torch.cuda.set_device(rank)
        device = torch.device(f"cuda:{rank}")

        print(f"\n[DDP_RANK {rank}] Initialized with device={device}, MASTER_PORT={port}")
        print(f"[DDP_RANK {rank}] Building model = {MODEL_NAME}, dataset = {DATASET_NAME} ...")

        # 1) Build the model
        net = get_model(
            MODEL_NAME,
            build=True,
            dataset=DATASET_NAME,
            dropout=dropout_rate,
            ignore_flag=False
        ).to(device)

        ddp_net = DDP(net, device_ids=[rank], output_device=rank)

        # 2) Build the dataset with distributed=True
        loader_params = dict(
            batch_size=batch_size,
            shuffle=False,
            num_workers=2
        )
        dataset = get_dataset(
            DATASET_NAME,
            build=True,
            transform_parameters=ddp_net.module,
            loader_parameters=loader_params,
            device="cpu",
            distributed=True
        )
        print(f"[DDP_RANK {rank}] Dataset built successfully.")

        # 3) Create optimizer
        optimizer = torch.optim.Adam(ddp_net.parameters(), lr=lr, weight_decay=0)

        # 4) Train
        train_params = dict(
            num_epochs=epochs,
            alignment=True,
            alignment_expansion=False,
            compare_expected=False,
            frequency=1,
            delta_alignment=False,
        )
        nets = [ddp_net]
        optimizers = [optimizer]

        print(f"[DDP_RANK {rank}] Starting training for {epochs} epochs.")
        results = train.train(nets, optimizers, dataset, **train_params)
        print(f"[DDP_RANK {rank}] Training complete. Results keys = {list(results.keys())}")

        # 5) Progressive Dropout on rank 0
        if rank == 0:
            print("[DDP_RANK 0] Doing progressive dropout experiment...")
            dropout_params = {
                "num_drops": 3,
                "by_layer": False,
                "train_set": False,
            }
            if "alignment" not in results:
                print("[DDP_RANK 0] No 'alignment' in results - skipping.")
            else:
                dropout_res = progressive_dropout(nets, dataset, alignment=results["alignment"], **dropout_params)
                print("[DDP_RANK 0] Dropout results keys:", list(dropout_res.keys()))
                # Plot
                plotting.plot_dropout_results(
                    exp=None,
                    dropout_results=dropout_res,
                    dropout_parameters=dropout_params,
                    prms={
                        "vals": [MODEL_NAME],
                        "name": "ModelType",
                        "dataset": DATASET_NAME,
                        "dropout": dropout_rate,
                        "lr": lr,
                        "weight_decay": 0
                    },
                    dropout_type="alignment-based",
                )

    except Exception as e:
        print(f"\n[DDP_RANK {rank}] ERROR! => {e}")
        traceback.print_exc()
        # Reraise to let mp.spawn see the error and fail
        raise e
    finally:
        dist.destroy_process_group()
        print(f"[DDP_RANK {rank}] Done. Cleaned up process group.")

def run_ddp_experiment(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    dropout_rate=DROPOUT_RATE,
    n_gpus=NUM_GPUS
):
    # Pick a random port to avoid collisions
    import random
    port = random.randint(20000, 30000)

    print(f"Running DDP with n_gpus={n_gpus}, port={port}, epochs={epochs}, batch_size={batch_size}.")
    mp.spawn(
        ddp_worker,
        nprocs=n_gpus,
        args=(n_gpus, port, epochs, batch_size, lr, dropout_rate),
        join=True
    )
    print("DDP experiment finished.")

# Finally, call run_ddp_experiment. 
# This must be guarded if inside a Jupyter cell to avoid re-entry on child processes.
if __name__ == "__main__":
    run_ddp_experiment()

In [ ]:
# # Cell 3: Actually run the multi-GPU experiment from the notebook

# if __name__ == "__main__":
#     # Ensure the cell doesn't run repeatedly on each spawned process
#     # (mp.spawn re-imports the notebook code).
#     # We'll do a safe guard: only rank=0 calls it. But let's do a check:
#     # We'll just call run_ddp_experiment once.

#     # You can change these if you have fewer or more GPUs:
#     N_GPUS = NUM_GPUS  # 4 if you have 4 available
#     EPOCHS = EPOCHS
#     BATCH = BATCH_SIZE
#     LR = LEARNING_RATE
#     DROPOUT = DROPOUT_RATE

#     run_ddp_experiment(epochs=EPOCHS, batch_size=BATCH, lr=LR, dropout_rate=DROPOUT, n_gpus=N_GPUS)